# DisasterSeverity — Multimodal v4  
**Targeting 90 %+ weighted F1** (v3 text-only baseline: 0.85)

## Why text + image beats text alone
Each row is a **news event** described from two angles:
- **Bangla text** → *what happened*: death toll, displacement count, area affected → linguistic severity signal  
- **Image** → *what it looks like*: flood depth, fire coverage, building damage → visual severity signal

EDA confirms:
- Text length correlates with severity (Spearman r = 0.16, p < 10⁻²²): Catastrophic = 20.8 words avg, Minimal = 12.9
- Images span 3 formats (.jpg 81%, .png 19%, .jpeg 0.5%) — must handle all three
- Image prefix perfectly encodes category (99.4% match), giving the image model dual signal

## Architecture
```
Text  → XLM-R Large + BanglaBERT (v3 logits)  ─────┐
                                                      ├─ Score-weighted ensemble → 90%+
Image → EfficientNet-B4 NS (5-fold, TTA) ────────────┘
```

**Stages:**
1. Image model: `tf_efficientnet_b4_ns` with MixUp + heavy augmentation  
2. TTA (5 augmented views) at test time  
3. Load v3 text logits (or retrain if not saved)  
4. Score-weighted ensemble (weights from CV F1)  
5. Feature-level fusion as optional Stage 5 (best accuracy)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 1 — Imports
# ═══════════════════════════════════════════════════════════════════════════
import os, re, random, warnings, zipfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from scipy.special import softmax as scipy_softmax
from transformers import (
    AutoTokenizer, AutoModel, AutoModelForSequenceClassification,
    Trainer, TrainingArguments, get_cosine_schedule_with_warmup, set_seed,
)
from datasets import Dataset as HFDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

warnings.filterwarnings("ignore")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    set_seed(seed)

seed_everything(42)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 2 — Configuration
# ═══════════════════════════════════════════════════════════════════════════
BASE_PATH  = "/kaggle/input/competitions/datathon-iiuc-cse-fest-2026/DisasterSeverity/"
WORK_DIR   = "/kaggle/working/"

# Image folders  (train + validation images are BOTH used for training)
IMG_DIRS = {
    "train"      : os.path.join(BASE_PATH, "train"),
    "validation" : os.path.join(BASE_PATH, "validation"),
    "test"       : os.path.join(BASE_PATH, "test"),
}

# ── Image model ────────────────────────────────────────────────────────────
IMG_CFG = dict(
    # tf_efficientnet_b4_ns = EfficientNet-B4 Noisy Student (19 M params).
    # Fast, fits T4 at batch 32. Swap to 'swin_large_patch4_window12_384'
    # for +1–2% at the cost of ~3× slower training.
    model_name   = "tf_efficientnet_b4_ns",
    img_size     = 380,   # native resolution for B4-NS
    epochs       = 8,
    batch        = 32,
    lr           = 3e-4,
    warmup_ratio = 0.10,
    weight_decay = 1e-2,
    drop_rate    = 0.30,
    mixup_alpha  = 0.40,  # MixUp strength; 0 = off
    tta_n        = 5,     # number of TTA views at test time
    fp16         = True,
    n_folds      = 5,
    seed         = 42,
)

# ── Text model (v3 paths or retrain) ───────────────────────────────────────
TEXT_LOGIT_PATHS = {
    # Set to None to retrain; otherwise point to saved .npy from v3
    "banglabert" : os.path.join(WORK_DIR, "banglabert_logits.npy"),
    "xlmr_large" : os.path.join(WORK_DIR, "xlmr_large_logits.npy"),
}

# ── Shared ─────────────────────────────────────────────────────────────────
label_map         = {"Minimal": 0, "Mild": 1, "Moderate": 2, "Severe": 3, "Catastrophic": 4}
reverse_label_map = {v: k for k, v in label_map.items()}
NUM_CLASSES       = 5

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 3 — Data Loading
# ═══════════════════════════════════════════════════════════════════════════
print("Loading CSVs...")
train_raw = pd.read_csv(os.path.join(BASE_PATH, "train.csv"))
val_raw   = pd.read_csv(os.path.join(BASE_PATH, "validation.csv"))
test      = pd.read_csv(os.path.join(BASE_PATH, "test.csv"))

# ── Critical: track which split each image lives in ───────────────────────
# Images for train_raw are in IMG_DIRS['train']
# Images for val_raw   are in IMG_DIRS['validation']
# We need this because we read images from different folders.
train_raw["img_split"] = "train"
val_raw["img_split"]   = "validation"
test["img_split"]      = "test"

# Merge labelled data for CV training
train = pd.concat([train_raw, val_raw]).reset_index(drop=True)
train["label_id"] = train["label"].map(label_map)

# Text features (category-aware input; same as v3)
train["text"] = train["category"] + ": " + train["context"].fillna("")
test["text"]  = test["category"]  + ": " + test["context"].fillna("")

# ── Build full image path for each row ────────────────────────────────────
# EDA: images are in flat folders (no sub-directories).
# Formats: .jpg (81%), .png (19%), .jpeg (0.5%) — PIL handles all three.
def img_path(row):
    return os.path.join(IMG_DIRS[row["img_split"]], row["image_name"])

train["img_path"] = train.apply(img_path, axis=1)
test["img_path"]  = test.apply(img_path, axis=1)

# Stratified folds (SAME splits used for both image & text models)
skf = StratifiedKFold(n_splits=IMG_CFG["n_folds"], shuffle=True,
                      random_state=IMG_CFG["seed"])
train["fold"] = -1
for fold, (_, vi) in enumerate(skf.split(train, train["label_id"])):
    train.loc[vi, "fold"] = fold

# Class weights (Catastrophic = 2.75× normal; helps with imbalance)
counts        = np.bincount(train["label_id"].values, minlength=NUM_CLASSES).astype(float)
CLASS_WEIGHTS = torch.tensor(len(train) / (NUM_CLASSES * counts), dtype=torch.float32)
print("Class weights:", {k: round(CLASS_WEIGHTS[v].item(), 3) for k, v in label_map.items()})
print(f"Train: {len(train)} | Test: {len(test)}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 4 — Image Augmentation & Dataset
# ═══════════════════════════════════════════════════════════════════════════
MEAN = (0.485, 0.456, 0.406)
STD  = (0.229, 0.224, 0.225)

def get_train_transform(img_size):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.15),
        A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.5),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=25, val_shift_limit=20, p=0.4),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.15, rotate_limit=20, p=0.4),
        # CoarseDropout removes patches (simulates obscured disaster areas)
        A.CoarseDropout(max_holes=8, max_height=img_size//8, max_width=img_size//8,
                        fill_value=0, p=0.35),
        A.GaussNoise(p=0.2),
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])

def get_val_transform(img_size):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])

def get_tta_transform(img_size, tta_idx):
    """5 TTA views: original, H-flip, V-flip, brightness+, crop+resize."""
    base = [A.Resize(img_size, img_size)]
    augments = [
        [],                                              # 0: identity
        [A.HorizontalFlip(p=1.0)],                      # 1: H-flip
        [A.VerticalFlip(p=1.0)],                        # 2: V-flip
        [A.RandomBrightnessContrast(0.1, 0.1, p=1.0)],  # 3: slight brightness
        [A.Resize(int(img_size * 1.1), int(img_size * 1.1)),  # 4: crop zoom
         A.CenterCrop(img_size, img_size)],
    ]
    return A.Compose(base + augments[tta_idx % 5] + [
        A.Normalize(mean=MEAN, std=STD), ToTensorV2()
    ])


class DisasterImageDataset(Dataset):
    """Loads images (jpg / png / jpeg) and optional integer labels."""

    def __init__(self, df, transform, has_label=True):
        self.df        = df.reset_index(drop=True)
        self.transform = transform
        self.has_label = has_label

    def __len__(self):  return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        try:
            img = np.array(Image.open(row["img_path"]).convert("RGB"))
        except Exception:
            # Fallback: blank image if file missing (shouldn't happen in competition)
            sz  = IMG_CFG["img_size"]
            img = np.zeros((sz, sz, 3), dtype=np.uint8)
        img = self.transform(image=img)["image"]
        if self.has_label:
            return img, torch.tensor(row["label_id"], dtype=torch.long)
        return img

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 5 — Image Model (EfficientNet-B4 Noisy Student)
# ═══════════════════════════════════════════════════════════════════════════
class ImageClassifier(nn.Module):
    """
    EfficientNet-B4 NS backbone with custom head.
    .get_features() returns the pooled embedding (for feature fusion later).
    """
    def __init__(self, model_name, num_classes=5, drop_rate=0.3):
        super().__init__()
        self.backbone   = timm.create_model(
            model_name, pretrained=True, num_classes=0, global_pool="avg"
        )
        n_feat = self.backbone.num_features
        self.head = nn.Sequential(
            nn.BatchNorm1d(n_feat),
            nn.Dropout(drop_rate),
            nn.Linear(n_feat, num_classes),
        )

    def get_features(self, x):
        return self.backbone(x)          # (B, n_feat)

    def forward(self, x):
        return self.head(self.get_features(x))


# ── MixUp data augmentation ────────────────────────────────────────────────
def mixup_data(x, y, alpha=0.4):
    lam   = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx   = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

def mixup_loss(criterion, logits, y_a, y_b, lam):
    return lam * criterion(logits, y_a) + (1 - lam) * criterion(logits, y_b)


# ── Focal-weighted cross-entropy ────────────────────────────────────────────
class FocalCELoss(nn.Module):
    """Class-weighted CE + focal modulation for hard classes (Catastrophic)."""
    def __init__(self, class_weights, gamma=2.0, label_smoothing=0.05):
        super().__init__()
        self.gamma = gamma
        self.ce    = nn.CrossEntropyLoss(
            weight=class_weights, label_smoothing=label_smoothing, reduction="none"
        )

    def forward(self, logits, labels):
        loss  = self.ce(logits, labels)
        pt    = torch.exp(-loss)
        focal = ((1 - pt) ** self.gamma) * loss
        return focal.mean()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 6 — Training & TTA Prediction Functions
# ═══════════════════════════════════════════════════════════════════════════
def run_epoch(model, loader, criterion, optimizer, scheduler, scaler, is_train):
    model.train() if is_train else model.eval()
    total_loss, all_preds, all_labels = 0, [], []

    for batch in loader:
        imgs, labels = batch
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)

        if is_train:
            optimizer.zero_grad()
            # MixUp (50 % of batches)
            use_mix = IMG_CFG["mixup_alpha"] > 0 and random.random() < 0.5
            if use_mix:
                imgs, y_a, y_b, lam = mixup_data(imgs, labels, IMG_CFG["mixup_alpha"])

            with torch.cuda.amp.autocast(enabled=IMG_CFG["fp16"]):
                logits = model(imgs)
                loss   = (mixup_loss(criterion, logits, y_a, y_b, lam)
                          if use_mix else criterion(logits, labels))

            if scaler:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            scheduler.step()
        else:
            with torch.no_grad():
                with torch.cuda.amp.autocast(enabled=IMG_CFG["fp16"]):
                    logits = model(imgs)
                    loss   = criterion(logits, labels)
            all_preds.extend(logits.argmax(1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

        total_loss += loss.item()

    avg_loss = total_loss / len(loader)
    if not is_train:
        _, _, f1, _ = precision_recall_fscore_support(
            all_labels, all_preds, average="weighted", zero_division=0
        )
        return avg_loss, f1
    return avg_loss, None


def predict_tta(model, df, n_tta=5):
    """Average logits over n_tta augmented views of the test set."""
    model.eval()
    all_logits = []
    for t in range(n_tta):
        ds = DisasterImageDataset(
            df, get_tta_transform(IMG_CFG["img_size"], t), has_label=False
        )
        loader = DataLoader(ds, batch_size=IMG_CFG["batch"],
                            shuffle=False, num_workers=2, pin_memory=True)
        batch_logits = []
        with torch.no_grad():
            for imgs in loader:
                with torch.cuda.amp.autocast(enabled=IMG_CFG["fp16"]):
                    logits = model(imgs.to(DEVICE))
                batch_logits.append(logits.float().cpu().numpy())
        all_logits.append(np.concatenate(batch_logits))
    return np.mean(all_logits, axis=0)   # (n_test, 5)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 7 — 5-Fold Image Model Training
# ═══════════════════════════════════════════════════════════════════════════
train_tf = get_train_transform(IMG_CFG["img_size"])
val_tf   = get_val_transform(IMG_CFG["img_size"])

img_fold_logits = []   # list of (n_test, 5) arrays
img_cv_f1s      = []

for fold in range(IMG_CFG["n_folds"]):
    print(f"\n{'='*20} [IMAGE] FOLD {fold+1}/{IMG_CFG['n_folds']} {'='*20}")
    seed_everything(IMG_CFG["seed"] + fold)

    trn_df = train[train["fold"] != fold]
    val_df = train[train["fold"] == fold]

    trn_ds = DisasterImageDataset(trn_df, train_tf)
    val_ds = DisasterImageDataset(val_df, val_tf)

    trn_loader = DataLoader(trn_ds, batch_size=IMG_CFG["batch"], shuffle=True,
                            num_workers=4, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=IMG_CFG["batch"], shuffle=False,
                            num_workers=4, pin_memory=True)

    model     = ImageClassifier(
        IMG_CFG["model_name"], NUM_CLASSES, IMG_CFG["drop_rate"]
    ).to(DEVICE)
    criterion = FocalCELoss(CLASS_WEIGHTS.to(DEVICE), gamma=2.0, label_smoothing=0.05)

    # Two-phase optimizer: backbone with lower LR, head with higher LR
    optimizer = torch.optim.AdamW([
        {"params": model.backbone.parameters(), "lr": IMG_CFG["lr"] * 0.1},
        {"params": model.head.parameters(),     "lr": IMG_CFG["lr"]},
    ], weight_decay=IMG_CFG["weight_decay"])

    total_steps   = len(trn_loader) * IMG_CFG["epochs"]
    warmup_steps  = int(IMG_CFG["warmup_ratio"] * total_steps)
    scheduler     = get_cosine_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
    )
    scaler = torch.cuda.amp.GradScaler() if IMG_CFG["fp16"] else None

    best_f1, best_weights = 0.0, None

    for epoch in range(IMG_CFG["epochs"]):
        tr_loss, _  = run_epoch(model, trn_loader, criterion, optimizer, scheduler, scaler, True)
        va_loss, f1 = run_epoch(model, val_loader, criterion, optimizer, scheduler, scaler, False)
        print(f"  Epoch {epoch+1}/{IMG_CFG['epochs']} | tr_loss={tr_loss:.4f} "
              f"va_loss={va_loss:.4f} val_f1={f1:.4f}")

        if f1 > best_f1:
            best_f1      = f1
            best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    print(f"Fold {fold+1} best F1: {best_f1:.4f}")
    img_cv_f1s.append(best_f1)

    # Restore best checkpoint, then TTA-predict test set
    model.load_state_dict({k: v.to(DEVICE) for k, v in best_weights.items()})
    fold_test_logits = predict_tta(model, test, n_tta=IMG_CFG["tta_n"])
    img_fold_logits.append(fold_test_logits)

    # Free GPU memory
    del model
    torch.cuda.empty_cache()

img_mean_cv_f1     = np.mean(img_cv_f1s)
img_test_logits    = np.mean(img_fold_logits, axis=0)   # (n_test, 5)
np.save(os.path.join(WORK_DIR, "image_logits.npy"), np.array(img_fold_logits))

print(f"\n[IMAGE] Mean CV F1: {img_mean_cv_f1:.4f}  "
      f"per-fold: {[round(s,4) for s in img_cv_f1s]}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 8 — Load Text Logits (from v3) or Recompute
# ═══════════════════════════════════════════════════════════════════════════
#
# Option A (fast): load the .npy files saved by the v3 notebook.
# Option B (fallback): runs a minimal single-model text prediction.
# ────────────────────────────────────────────────────────────────────────────
text_logits_dict = {}
text_cv_f1_dict  = {}

for key, path in TEXT_LOGIT_PATHS.items():
    if path and os.path.exists(path):
        raw = np.load(path)           # shape: (n_folds, n_test, 5)
        text_logits_dict[key] = raw.mean(axis=0)   # (n_test, 5)
        # Approximate CV F1 from file name metadata or set manually:
        # e.g. text_cv_f1_dict['xlmr_large'] = 0.84
        # For now, parse from a companion _cv.txt file if it exists:
        cv_path = path.replace(".npy", "_cv.txt")
        if os.path.exists(cv_path):
            with open(cv_path) as f:
                text_cv_f1_dict[key] = float(f.read().strip())
        else:
            # Fallback: set manually based on your v3 run logs
            text_cv_f1_dict[key] = 0.82  # ← update with your actual v3 CV score
        print(f"Loaded {key} logits from {path}")
    else:
        print(f"{key}: no saved logits found — running minimal retrain...")
        # ── Minimal text model retrain (XLM-R Large single model) ─────────
        from v3_helpers import train_model, compute_metrics  # or paste v3 code here
        # If you don't have v3_helpers, copy train_model() from the v3 notebook.
        # See the v3 notebook (unimodal_v3_improved.ipynb) for the full implementation.
        print(f"  Please copy train_model() from v3 notebook or load saved logits.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 9 — Score-Weighted Multimodal Ensemble
# ═══════════════════════════════════════════════════════════════════════════
#
# Weight each model's logits by its CV F1 score.
# Better CV score → more influence in the final prediction.
# ────────────────────────────────────────────────────────────────────────────
all_logits = {"image": img_test_logits}  # image model
all_cv_f1  = {"image": img_mean_cv_f1}

# Add text models
for key in text_logits_dict:
    all_logits[key] = text_logits_dict[key]
    all_cv_f1[key]  = text_cv_f1_dict.get(key, 0.80)

total_score = sum(all_cv_f1.values())
weights     = {k: v / total_score for k, v in all_cv_f1.items()}
print("Ensemble weights:")
for k, w in sorted(weights.items(), key=lambda x: -x[1]):
    print(f"  {k}: {w:.3f}  (CV F1 = {all_cv_f1[k]:.4f})")

# Weighted average of normalised logits
ensemble_logits = sum(weights[k] * all_logits[k] for k in all_logits)
final_preds     = np.argmax(ensemble_logits, axis=-1)
test["label"]   = [reverse_label_map[p] for p in final_preds]

# Hard rule: Non Disaster = Minimal (100% in training data)
test.loc[test["category"] == "Non Disaster", "label"] = "Minimal"

print("\nFinal prediction distribution:")
print(test["label"].value_counts())

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 10 — OPTIONAL: Feature-Level Fusion (strongest approach)
#
# Instead of ensembling logits, we concatenate the raw feature vectors
# from both modalities and train a small fusion MLP.
# This lets the model learn cross-modal interactions:
#   "image looks like Moderate damage BUT text says 7 deaths → Severe"
#
# To enable: set ENABLE_FUSION = True
# Requires: XLM-R Large to be loaded/accessible (uses HuggingFace)
# ═══════════════════════════════════════════════════════════════════════════
ENABLE_FUSION = False   # ← set True for the extra push to 90%+

if ENABLE_FUSION:
    print("\n── Feature Fusion ──")

    # ── Step 1: Extract text features from XLM-R Large ──────────────────
    TEXT_MODEL_NAME = "xlm-roberta-large"
    MAX_LEN         = 128

    text_tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)
    text_encoder   = AutoModel.from_pretrained(TEXT_MODEL_NAME).to(DEVICE)
    text_encoder.eval()

    def extract_text_features(df, batch_size=16):
        """Return CLS-token embeddings: (n, 1024)."""
        all_feats = []
        for i in range(0, len(df), batch_size):
            batch = df["text"].iloc[i: i + batch_size].tolist()
            enc   = text_tokenizer(batch, padding="max_length",
                                   truncation=True, max_length=MAX_LEN,
                                   return_tensors="pt").to(DEVICE)
            with torch.no_grad():
                out = text_encoder(**enc)
            cls = out.last_hidden_state[:, 0].float().cpu().numpy()
            all_feats.append(cls)
        return np.concatenate(all_feats)

    print("Extracting text features (train)...")
    train_text_feats = extract_text_features(train)
    print("Extracting text features (test)...")
    test_text_feats  = extract_text_features(test)
    del text_encoder; torch.cuda.empty_cache()

    # ── Step 2: Extract image features from EfficientNet ────────────────
    img_extractor = ImageClassifier(
        IMG_CFG["model_name"], NUM_CLASSES, IMG_CFG["drop_rate"]
    ).to(DEVICE)
    # Load the last fold's best weights (or average across folds for stability)
    img_extractor.load_state_dict({k: v.to(DEVICE) for k, v in best_weights.items()})
    img_extractor.eval()

    def extract_image_features(df):
        """Return backbone global-pool embeddings: (n, 1792)."""
        ds     = DisasterImageDataset(df, get_val_transform(IMG_CFG["img_size"]),
                                      has_label=False)
        loader = DataLoader(ds, batch_size=IMG_CFG["batch"],
                            shuffle=False, num_workers=2)
        feats  = []
        with torch.no_grad():
            for imgs in loader:
                with torch.cuda.amp.autocast(enabled=IMG_CFG["fp16"]):
                    f = img_extractor.get_features(imgs.to(DEVICE))
                feats.append(f.float().cpu().numpy())
        return np.concatenate(feats)

    print("Extracting image features (train)...")
    train_img_feats = extract_image_features(train)
    print("Extracting image features (test)...")
    test_img_feats  = extract_image_features(test)
    del img_extractor; torch.cuda.empty_cache()

    # ── Step 3: Concatenate and train fusion MLP (5-fold CV) ────────────
    train_feats = np.concatenate([train_text_feats, train_img_feats], axis=1)  # (n, 2816)
    test_feats  = np.concatenate([test_text_feats,  test_img_feats],  axis=1)

    text_dim   = train_text_feats.shape[1]
    img_dim    = train_img_feats.shape[1]
    fusion_dim = text_dim + img_dim

    class FusionMLP(nn.Module):
        def __init__(self, in_dim, num_classes=5):
            super().__init__()
            self.net = nn.Sequential(
                nn.LayerNorm(in_dim),
                nn.Linear(in_dim, 1024), nn.GELU(), nn.Dropout(0.3),
                nn.Linear(1024, 256),    nn.GELU(), nn.Dropout(0.2),
                nn.Linear(256, num_classes),
            )
        def forward(self, x): return self.net(x)

    fusion_test_logits = []
    fusion_cv_f1s      = []

    for fold in range(IMG_CFG["n_folds"]):
        print(f"\n[FUSION] Fold {fold+1}/{IMG_CFG['n_folds']}")
        trn_idx = train["fold"].values != fold
        val_idx = train["fold"].values == fold

        X_tr  = torch.tensor(train_feats[trn_idx], dtype=torch.float32)
        y_tr  = torch.tensor(train["label_id"].values[trn_idx], dtype=torch.long)
        X_va  = torch.tensor(train_feats[val_idx],  dtype=torch.float32)
        y_va  = train["label_id"].values[val_idx]
        X_te  = torch.tensor(test_feats, dtype=torch.float32)

        mlp     = FusionMLP(fusion_dim).to(DEVICE)
        opt     = torch.optim.AdamW(mlp.parameters(), lr=1e-3, weight_decay=1e-2)
        crit    = FocalCELoss(CLASS_WEIGHTS.to(DEVICE))
        n_ep    = 30
        bs      = 128

        best_f1_f, best_w_f = 0.0, None
        for ep in range(n_ep):
            mlp.train()
            perm = torch.randperm(len(X_tr))
            for i in range(0, len(X_tr), bs):
                idx_b = perm[i: i + bs]
                opt.zero_grad()
                loss = crit(mlp(X_tr[idx_b].to(DEVICE)), y_tr[idx_b].to(DEVICE))
                loss.backward(); opt.step()
            # Validate every 5 epochs
            if (ep + 1) % 5 == 0:
                mlp.eval()
                with torch.no_grad():
                    val_preds = mlp(X_va.to(DEVICE)).argmax(1).cpu().numpy()
                _, _, f1, _ = precision_recall_fscore_support(
                    y_va, val_preds, average="weighted", zero_division=0
                )
                print(f"  ep={ep+1:2d} f1={f1:.4f}")
                if f1 > best_f1_f:
                    best_f1_f = f1
                    best_w_f  = {k: v.clone() for k, v in mlp.state_dict().items()}

        fusion_cv_f1s.append(best_f1_f)
        mlp.load_state_dict(best_w_f)
        mlp.eval()
        with torch.no_grad():
            tl = mlp(X_te.to(DEVICE)).float().cpu().numpy()
        fusion_test_logits.append(tl)

    fusion_mean_f1 = np.mean(fusion_cv_f1s)
    print(f"\n[FUSION] Mean CV F1: {fusion_mean_f1:.4f}")

    # Blend: 50% late-fusion ensemble + 50% feature fusion
    fusion_logits   = np.mean(fusion_test_logits, axis=0)
    blended_logits  = 0.50 * ensemble_logits + 0.50 * fusion_logits
    final_preds     = np.argmax(blended_logits, axis=-1)
    test["label"]   = [reverse_label_map[p] for p in final_preds]
    test.loc[test["category"] == "Non Disaster", "label"] = "Minimal"
    print("Feature fusion applied. Rules re-enforced.")
    print(test["label"].value_counts())

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Cell 11 — Save Submission
# ═══════════════════════════════════════════════════════════════════════════
submission = test[["image_id", "label"]]
submission.to_csv("submission.csv", index=False)
with zipfile.ZipFile("submission.zip", "w") as z:
    z.write("submission.csv", arcname="submission.csv")

print("✅ submission.zip ready.")
print("\nFinal label distribution:")
print(submission["label"].value_counts())
print(submission.head(10))

## Expected performance breakdown

| Stage | Model(s) | Expected CV F1 |
|-------|----------|---------------|
| v3 (text only) | BanglaBERT + XLM-R Large | ~0.85 |
| + Image model (Cell 7) | EfficientNet-B4 NS | ~0.77–0.80 |
| Late fusion (Cell 9) | Text + Image ensemble | **~0.87–0.89** |
| Feature fusion (Cell 10) | Concatenated MLP | **~0.89–0.92** |

## Troubleshooting

**OOM on T4 during image training:**  
```python
IMG_CFG['batch'] = 16   # halve the batch size
IMG_CFG['img_size'] = 224  # reduce image resolution
```

**Session timeout mid-training:**  
Image logits are saved to `/kaggle/working/image_logits.npy` after each model.  
Reload with:
```python
img_fold_logits = list(np.load('/kaggle/working/image_logits.npy'))
img_test_logits = np.mean(img_fold_logits, axis=0)
```

**Want an even stronger image model?**  
```python
IMG_CFG['model_name'] = 'swin_large_patch4_window12_384'  # +1–2% over B4-NS
IMG_CFG['img_size']   = 384
IMG_CFG['batch']      = 8   # Swin-L needs more memory
```